# Fabric acceptance and export
Checks persisted Delta Tables, writes reports and a PASS marker only after all checks succeed.

In [ ]:
run_id = 'manual'
source_folder = 'Source'
bronze_folder = 'Bronze'
reports_folder = 'Reports'
table_schema = 'dbo'
strict_project_snapshot = True

In [ ]:
import re
from pyspark.sql import functions as F, types as T
assert re.fullmatch(r'[A-Za-z0-9_-]+', run_id), 'Invalid run_id'
assert not table_schema or re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',table_schema)
def table_name(name):
    assert re.fullmatch(r'[a-z_]+',name)
    return f'{table_schema}.{name}' if table_schema else name
def table(name): return spark.table(table_name(name))
def save(df, name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(table_name(name))
def bronze_path(kind): return f'Files/{bronze_folder}/{run_id}/{kind}/'
def source_frame(filename):
    return (spark.read.option('header',True).option('inferSchema',False).option('quote','"')
        .option('escape','"').option('mode','FAILFAST').csv(f'Files/{source_folder}/{filename}'))

In [ ]:
import json
from datetime import datetime, timezone
from decimal import Decimal
expected_tables = ['silver_sales', 'silver_sales_items', 'silver_sales_quarantine', 'silver_exchange_rate', 'silver_exchange_rate_quarantine', 'gold_dim_date', 'gold_dim_customer', 'gold_dim_location', 'gold_dim_payment_method', 'gold_dim_product', 'gold_fact_orders', 'gold_fact_order_items', 'gold_monthly_revenue_category', 'gold_monthly_completed_revenue_category', 'gold_promotion_by_location']
counts = {name: table(name).count() for name in expected_tables}
o = table('silver_sales')
fo = table('gold_fact_orders')
fi = table('gold_fact_order_items')
assert counts['silver_sales']==counts['gold_fact_orders']
assert counts['silver_sales_items']==counts['gold_fact_order_items']
assert o.count()==o.select('order_id').distinct().count()
assert fi.groupBy('order_id','item_index').count().filter('count > 1').count()==0
assert fi.join(fo.select('order_id'),'order_id','left_anti').count()==0
for fk,dimension in [('date_key','gold_dim_date'),('customer_key','gold_dim_customer'),('location_key','gold_dim_location'),('payment_key','gold_dim_payment_method')]:
    d = table(dimension)
    assert d.count()==d.select(fk).distinct().count()
    assert fo.join(d.select(fk),fk,'left_anti').count()==0
assert fi.join(table('gold_dim_product').select('product_key'),'product_key','left_anti').count()==0
assert fi.join(table('gold_dim_date').select('date_key'),'date_key','left_anti').count()==0
eligible = fo.filter('report_eligible')
assert eligible.filter("order_status='Failed' OR feedback_score IS NULL OR feedback_score NOT BETWEEN 1 AND 5").count()==0
total = table('gold_monthly_revenue_category').agg(F.sum('total_revenue_vnd')).first()[0]
completed_total = table('gold_monthly_completed_revenue_category').agg(F.sum('total_revenue_vnd')).first()[0]
expected = eligible.select((F.col('total_amount_usd')*F.col('exchange_rate')).cast('decimal(30,6)').alias('v')).agg(F.sum('v')).first()[0]
assert total == expected
reconciled = fi.groupBy('order_id').agg(F.sum('allocated_revenue_usd').alias('allocated')).join(fo,'order_id')
assert reconciled.filter('allocated != total_amount_usd').count()==0
assert reconciled.count()==fo.count()
promo = table('gold_promotion_by_location')
assert promo.agg(F.sum('total_orders')).first()[0] == eligible.count()
assert promo.agg(F.sum('orders_with_discount')).first()[0] == eligible.filter('has_discount').count()
if strict_project_snapshot:
    assert counts['silver_sales']==4896 and counts['silver_sales_quarantine']==104
    assert counts['silver_sales_items']==9850 and counts['silver_exchange_rate']==24
    assert counts['silver_exchange_rate_quarantine']==0 and counts['gold_dim_date']==731
    assert eligible.count()==2669 and eligible.filter("order_status='Completed'").count()==885
    assert total==Decimal('68128580695.61')
    assert completed_total==Decimal('22440017306.84')
output_root = f'Files/{reports_folder}/{run_id}'
for name in ['gold_monthly_revenue_category','gold_monthly_completed_revenue_category','gold_promotion_by_location']:
    df=table(name)
    order=['year','month','category'] if 'revenue' in name else ['location']
    df.orderBy(*order).coalesce(1).write.mode('overwrite').option('header',True).csv(output_root+'/'+name)
# This marker is written last: reports are usable only after acceptance succeeds.
summary = {'status':'PASS','run_id':run_id,'engine':'Fabric PySpark','spark_version':spark.version,
  'finished_utc':datetime.now(timezone.utc).isoformat(),'table_counts':counts,
  'eligible_orders':eligible.count(),'completed_orders':eligible.filter("order_status='Completed'").count(),
  'revenue_vnd':str(total),'completed_revenue_vnd':str(completed_total)}
spark.range(1).select(F.lit(json.dumps(summary)).alias('value')).coalesce(1).write.mode('overwrite').text(output_root+'/acceptance')
print(json.dumps(summary,indent=2))